# XFeat-SuperPoint Hybrid — MegaDepth Raw Training (Hardened Colab)

This notebook is designed for **Google Colab GPU runtimes** and focuses on robust setup, safe extraction, reliable downloads, and visible training progress/metrics.


In [ ]:
#@title 1) Runtime preflight (GPU + Python + Colab + shell debug examples)
import os
import sys
import subprocess
from datetime import datetime, timezone

print('UTC now:', datetime.now(timezone.utc).isoformat(timespec='seconds'))

if 'COLAB_GPU' not in os.environ and 'google.colab' not in sys.modules:
    print('⚠️ This notebook is intended for Google Colab. Continuing anyway...')

smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if smi.returncode != 0:
    raise RuntimeError('No GPU runtime detected. In Colab use Runtime → Change runtime type → GPU.')

print(smi.stdout.splitlines()[0])
print('Python:', sys.version.split()[0])
print('Executable:', sys.executable)

print('\nShell command examples in notebooks:')
print('  - Single command: !python --version')
print('  - Multi-line bash: %%bash')
print('  - Programmatic: subprocess.run([...], check=True)')

# Debug cell payload executed correctly (no Python SyntaxError).
debug_cmd = 'CUDA_LAUNCH_BLOCKING=1 PYTHONFAULTHANDLER=1 python -c "import torch; print(\'debug-env-ok\', torch.cuda.is_available())"'
subprocess.run(['bash', '-lc', debug_cmd], check=True)


In [ ]:
#@title 2) Mount Google Drive
try:
    from google.colab import drive
except Exception as e:
    raise RuntimeError('google.colab is unavailable. Run this notebook in Google Colab.') from e

drive.mount('/content/drive', force_remount=False)


In [ ]:
#@title 3) Paths, refs, and training knobs
from pathlib import Path

# ===== User-configurable =====
DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/hybrid_feature_matching'
RAW_ROOT_DRIVE     = f'{DRIVE_PROJECT_ROOT}/data/megadepth_raw'   # scene dirs like 0001, 0002, ...
RAW_TAR_PATH       = ''  # optional tar on Drive, e.g. '/content/drive/MyDrive/.../megadepth_raw.tar'

REPO_URL            = 'https://github.com/MalharRane/XFeat-SuperPoint-Hybrid-Model.git'
REPO_BRANCH         = ''  # optional, e.g. 'main'. Empty = repo default branch.
REPO_DIR            = '/content/XFeat-SuperPoint-Hybrid-Model'
XFEAT_REPO_URL      = 'https://github.com/verlab/accelerated_features.git'
XFEAT_DIR           = '/content/accelerated_features'
SUPERPOINT_REPO_URL = 'https://github.com/rpautrat/SuperPoint.git'
SUPERPOINT_DIR      = '/content/SuperPoint'

# Pinned refs (required for reproducibility)
XFEAT_REF        = 'e92685f57f8318b18725c5c8c0bd28c7fe188d9a'
SUPERPOINT_REF   = '1411bbd68c50163555d39c1b26e9e046ebd48f27'
HYBRID_REPO_REF  = ''  # optional; leave empty to keep checked-out branch/commit
LIGHTGLUE_REF    = 'c91eb892b799fa607523879d18f9ec4fe76194a5'
ENABLE_LIGHTGLUE = True
ALLOW_LIGHTGLUE_DISABLE = True

DRIVE_CKPT_DIR      = f'{DRIVE_PROJECT_ROOT}/checkpoints/megadepth_raw'
DRIVE_LOG_DIR       = f'{DRIVE_PROJECT_ROOT}/runs/megadepth_raw'

# Training hyperparameters
BATCH_SIZE          = 4
MAX_EPOCHS          = 50
LR                  = 1e-4
NUM_WORKERS         = 0  # Colab-safe default
IMAGE_H, IMAGE_W    = 480, 640  # MUST be divisible by 8
VAL_SPLIT_RATIO     = 0.2
MAX_PAIRS_PER_SCENE = 1000
MIN_OVERLAP         = 0.15
MAX_OVERLAP         = 0.70
MIN_KEYPOINT_SCORE  = 0.01
SEED                = 42

# Resume checkpoint: leave empty to auto-pick best.pth / latest epoch_*.pth
RESUME_CKPT             = ''
NO_VERIFY_DATASET_PAIRS = False
VALIDATION_SCAN_LIMIT   = 250
OVERRIDE_CFG_PATH       = '/content/megadepth_raw_override.yaml'

Path(DRIVE_CKPT_DIR).mkdir(parents=True, exist_ok=True)
Path(DRIVE_LOG_DIR).mkdir(parents=True, exist_ok=True)

print('REPO_DIR       :', REPO_DIR)
print('RAW_ROOT_DRIVE :', RAW_ROOT_DRIVE)
print('DRIVE_CKPT_DIR :', DRIVE_CKPT_DIR)
print('DRIVE_LOG_DIR  :', DRIVE_LOG_DIR)
print('Pinned refs    :', {'XFEAT_REF': XFEAT_REF, 'SUPERPOINT_REF': SUPERPOINT_REF, 'HYBRID_REPO_REF': HYBRID_REPO_REF or '(none)', 'LIGHTGLUE_REF': LIGHTGLUE_REF})



In [ ]:
#@title 4) Clone/update repos + install pinned dependencies + provenance capture
import os
import sys
import time
import importlib
import subprocess
from pathlib import Path

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
os.environ.setdefault('TF_ENABLE_ONEDNN_OPTS', '0')


def run_with_retry(cmd, cwd=None, env=None, max_attempts=3, sleep_s=5, capture=False):
    last = None
    for attempt in range(1, max_attempts + 1):
        try:
            print('+', ' '.join(cmd))
            return subprocess.run(cmd, cwd=cwd, env=env, check=True, text=True, capture_output=capture)
        except subprocess.CalledProcessError as e:
            last = e
            if e.stdout:
                print('stdout tail\n', e.stdout[-2000:])
            if e.stderr:
                print('stderr tail\n', e.stderr[-2000:])
            if attempt < max_attempts:
                print(f'Command failed ({attempt}/{max_attempts}), retrying in {sleep_s}s...')
                time.sleep(sleep_s)
    raise last


def ensure_repo(path, url, branch=''):
    p = Path(path)
    if p.exists() and (p / '.git').exists():
        run_with_retry(['git', '-C', str(p), 'fetch', '--all', '--tags', '--prune'], max_attempts=3)
        if branch:
            run_with_retry(['git', '-C', str(p), 'checkout', branch], max_attempts=2)
            run_with_retry(['git', '-C', str(p), 'pull', '--ff-only', 'origin', branch], max_attempts=3)
        else:
            run_with_retry(['git', '-C', str(p), 'pull', '--ff-only'], max_attempts=3)
    elif p.exists():
        raise RuntimeError(f'Path exists but is not a git repo: {p}')
    else:
        clone_cmd = ['git', 'clone', url, str(p)]
        if branch:
            clone_cmd = ['git', 'clone', '--branch', branch, '--single-branch', url, str(p)]
        run_with_retry(clone_cmd, max_attempts=3)


def checkout_ref(repo_dir, ref, label):
    if not ref:
        return
    run_with_retry(['git', '-C', repo_dir, 'fetch', '--all', '--tags', '--prune'], max_attempts=3)
    run_with_retry(['git', '-C', repo_dir, 'checkout', ref], max_attempts=1)
    sha = subprocess.check_output(['git', '-C', repo_dir, 'rev-parse', 'HEAD'], text=True).strip()
    if not sha.startswith(ref[:7]):
        print(f'[{label}] checked out {sha} from ref {ref}')


def pip_install(args, max_attempts=3):
    return run_with_retry([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', *args], max_attempts=max_attempts)


def install_lightglue(enabled=True, allow_disable=True):
    if not enabled:
        print('MATCHER_BACKEND=disabled (by config)')
        return 'disabled'
    source = f'git+https://github.com/cvg/LightGlue.git@{LIGHTGLUE_REF}'
    try:
        pip_install([source], max_attempts=2)
        importlib.invalidate_caches()
        import lightglue
        if not hasattr(lightglue, 'LightGlue'):
            raise RuntimeError('lightglue.LightGlue symbol missing')
        print('MATCHER_BACKEND=lightglue')
        return 'lightglue'
    except Exception as e:
        if allow_disable:
            print(f'LightGlue unavailable, disabling matcher backend: {type(e).__name__}: {e}')
            print('MATCHER_BACKEND=disabled')
            return 'disabled'
        raise RuntimeError('LightGlue install/verification failed and disabling is not allowed') from e


def capture_provenance(repo_dir, xfeat_dir, superpoint_dir):
    out_dir = Path(repo_dir) / 'runs' / 'megadepth_raw'
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / 'env_provenance.txt'

    pyver = subprocess.check_output([sys.executable, '--version'], text=True).strip()
    pip_freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)

    import torch
    lines = []
    lines.append(pyver)
    lines.append(f'torch.__version__={torch.__version__}')
    lines.append(f'torch.version.cuda={torch.version.cuda}')

    for label, path in [
        ('MalharRane/XFeat-SuperPoint-Hybrid-Model', repo_dir),
        ('verlab/accelerated_features', xfeat_dir),
        ('rpautrat/SuperPoint', superpoint_dir),
    ]:
        sha = subprocess.check_output(['git', '-C', path, 'rev-parse', 'HEAD'], text=True).strip()
        lines.append(f'{label}={sha}')

    lines.append('\n[pip freeze]')
    lines.append(pip_freeze)
    out_path.write_text('\n'.join(lines))
    print('Provenance written to', out_path)


ensure_repo(REPO_DIR, REPO_URL, REPO_BRANCH.strip())
ensure_repo(XFEAT_DIR, XFEAT_REPO_URL)
ensure_repo(SUPERPOINT_DIR, SUPERPOINT_REPO_URL)

checkout_ref(XFEAT_DIR, XFEAT_REF, 'XFeat')
checkout_ref(SUPERPOINT_DIR, SUPERPOINT_REF, 'SuperPoint')
if HYBRID_REPO_REF:
    checkout_ref(REPO_DIR, HYBRID_REPO_REF, 'HybridRepo')

pip_install(['--upgrade', 'pip', 'setuptools', 'wheel'], max_attempts=3)
pip_install(['-r', f'{REPO_DIR}/requirements-colab.txt'], max_attempts=3)
backend = install_lightglue(enabled=ENABLE_LIGHTGLUE, allow_disable=ALLOW_LIGHTGLUE_DISABLE)

# Make repos importable in this runtime and subprocesses.
py_paths = [REPO_DIR, XFEAT_DIR, SUPERPOINT_DIR]
existing = os.environ.get('PYTHONPATH', '')
if existing:
    py_paths.extend(existing.split(os.pathsep))
py_paths = [p for p in dict.fromkeys(py_paths) if p]
os.environ['PYTHONPATH'] = os.pathsep.join(py_paths)

for p in reversed([REPO_DIR, XFEAT_DIR, SUPERPOINT_DIR]):
    if p not in sys.path:
        sys.path.insert(0, p)

# Quick import sanity for core libs.
for pkg in ['torch', 'numpy', 'yaml', 'h5py', 'kornia', 'matplotlib']:
    importlib.import_module(pkg)

capture_provenance(REPO_DIR, XFEAT_DIR, SUPERPOINT_DIR)
print('Dependency setup complete.')
print('PYTHONPATH=', os.environ['PYTHONPATH'])
print(f'Active matcher backend: {backend}')


In [ ]:
#@title 5) Download pretrained weights (idempotent + retry)
import time
import urllib.request
from pathlib import Path

weights_dir = Path(REPO_DIR) / 'weights'
weights_dir.mkdir(parents=True, exist_ok=True)

assets = [
    ('https://github.com/verlab/accelerated_features/releases/download/v1.0/xfeat.pt', weights_dir / 'xfeat.pth'),
    ('https://github.com/magicleap/SuperPointPretrainedNetwork/raw/master/superpoint_v1.pth', weights_dir / 'superpoint_v1.pth'),
]


def download_with_retry(url, dst, attempts=3, sleep_s=4):
    last = None
    for i in range(1, attempts + 1):
        try:
            print(f'Downloading ({i}/{attempts}) {url} -> {dst}')
            urllib.request.urlretrieve(url, str(dst))
            if dst.stat().st_size <= 0:
                raise RuntimeError(f'Empty file downloaded: {dst}')
            return
        except Exception as e:
            last = e
            if i < attempts:
                print(f'Download failed: {e}; retrying in {sleep_s}s...')
                time.sleep(sleep_s)
    raise RuntimeError(f'Failed to download {url}') from last


for url, dst in assets:
    if dst.exists() and dst.stat().st_size > 0:
        print('Exists:', dst, f'({dst.stat().st_size/1e6:.1f} MB)')
        continue
    download_with_retry(url, dst)

print('Weights ready in', weights_dir)


In [ ]:
#@title 6) Optional: safely extract RAW_TAR_PATH to /content/megadepth_raw
import tarfile
from pathlib import Path


def _safe_within_directory(base: Path, target: Path) -> bool:
    base_r = base.resolve()
    target_r = target.resolve()
    try:
        target_r.relative_to(base_r)
        return True
    except ValueError:
        return False


def safe_extract(tar: tarfile.TarFile, extract_root: Path):
    for member in tar.getmembers():
        normalized = Path(member.name)
        if normalized.is_absolute() or '..' in normalized.parts:
            raise RuntimeError(f'Unsafe tar entry (invalid path): {member.name}')
        member_path = extract_root / member.name
        if not _safe_within_directory(extract_root, member_path):
            raise RuntimeError(f'Unsafe tar entry (path traversal): {member.name}')
        if member.issym() or member.islnk():
            raise RuntimeError(f'Unsafe tar entry (links not allowed): {member.name}')
    tar.extractall(extract_root)


if RAW_TAR_PATH:
    tar_path = Path(RAW_TAR_PATH)
    if not tar_path.exists():
        raise FileNotFoundError(f'RAW_TAR_PATH not found: {tar_path}')

    extract_root = Path('/content/megadepth_raw')
    extraction_flag = extract_root / '.extracted'

    if extraction_flag.exists():
        print('Already extracted:', extract_root)
    else:
        extract_root.mkdir(parents=True, exist_ok=True)
        print('Extracting', tar_path, '->', extract_root)
        with tarfile.open(tar_path, 'r:*') as tf:
            safe_extract(tf, extract_root)
        if not any(extract_root.glob('**/dense0/imgs/*')):
            raise RuntimeError('Extraction completed but no images found under **/dense0/imgs')
        extraction_flag.write_text('ok')

    RAW_ROOT = str(extract_root)
else:
    RAW_ROOT = RAW_ROOT_DRIVE

print('RAW_ROOT:', RAW_ROOT)


In [ ]:
#@title 7) Validate MegaDepth raw layout + write dataset report
import json
import hashlib
import random
from pathlib import Path
import numpy as np
import h5py
from PIL import Image

raw = Path(RAW_ROOT)
if not raw.exists():
    raise FileNotFoundError(f'RAW_ROOT does not exist: {raw}')

scene_dirs = sorted([p for p in raw.iterdir() if p.is_dir()])
img_dirs = sorted(raw.glob('**/dense0/imgs'))
depth_dirs = sorted(raw.glob('**/dense0/depths'))

img_paths = []
for d in img_dirs:
    img_paths.extend(sorted(list(d.glob('*.jpg')) + list(d.glob('*.jpeg')) + list(d.glob('*.png'))))

depth_paths = []
for d in depth_dirs:
    depth_paths.extend(sorted(d.glob('*.h5')))

sample_images = random.sample(img_paths, k=min(len(img_paths), max(20, min(VALIDATION_SCAN_LIMIT, 200)))) if img_paths else []
sample_depths = random.sample(depth_paths, k=min(len(depth_paths), max(20, min(VALIDATION_SCAN_LIMIT, 200)))) if depth_paths else []

corrupt_images = []
for p in sample_images:
    try:
        with Image.open(p) as im:
            im.verify()
    except Exception:
        corrupt_images.append(str(p))

bad_depth_files = []
for p in sample_depths:
    try:
        with h5py.File(p, 'r') as f:
            if 'depth' not in f:
                bad_depth_files.append(f"{p} (missing 'depth' key)")
    except Exception as e:
        bad_depth_files.append(f'{p} ({type(e).__name__}: {e})')

# Overlap sanity proxy range from adjacent frame thumbnails (NCC mapped to [0,1]).
def thumb(path, size=64):
    arr = np.asarray(Image.open(path).convert('L').resize((size, size), Image.BILINEAR), dtype=np.float32) / 255.0
    return arr

def overlap_proxy(a, b):
    a0 = a - a.mean()
    b0 = b - b.mean()
    denom = float(np.sqrt((a0 * a0).sum() * (b0 * b0).sum()) + 1e-8)
    corr = float((a0 * b0).sum() / denom)
    return max(0.0, min(1.0, 0.5 * (corr + 1.0)))

ov_vals = []
for d in img_dirs[:VALIDATION_SCAN_LIMIT]:
    local = sorted(list(d.glob('*.jpg')) + list(d.glob('*.jpeg')) + list(d.glob('*.png')))
    if len(local) < 2:
        continue
    for i in range(min(len(local) - 1, 5)):
        try:
            ov_vals.append(overlap_proxy(thumb(local[i]), thumb(local[i + 1])))
        except Exception:
            continue

# Train/val split disjointness report using deterministic scene hashing.
def is_val(scene_id, ratio):
    digest = hashlib.sha256(scene_id.encode('utf-8')).hexdigest()
    bucket = int(digest[:8], 16) / float(16**8)
    return bucket < ratio

scene_ids = [p.name for p in scene_dirs]
train_scene_ids = [s for s in scene_ids if not is_val(s, VAL_SPLIT_RATIO)]
val_scene_ids = [s for s in scene_ids if is_val(s, VAL_SPLIT_RATIO)]
intersection = sorted(set(train_scene_ids).intersection(val_scene_ids))

report = {
    'raw_root': str(raw),
    'scene_count': len(scene_dirs),
    'img_dir_count': len(img_dirs),
    'depth_dir_count': len(depth_dirs),
    'image_count': len(img_paths),
    'depth_count': len(depth_paths),
    'sampled_image_validation': {
        'sample_size': len(sample_images),
        'corrupt_count': len(corrupt_images),
        'corrupt_examples': corrupt_images[:10],
    },
    'sampled_depth_validation': {
        'sample_size': len(sample_depths),
        'bad_count': len(bad_depth_files),
        'bad_examples': bad_depth_files[:10],
    },
    'overlap_sanity': {
        'sample_count': len(ov_vals),
        'min': float(min(ov_vals)) if ov_vals else None,
        'max': float(max(ov_vals)) if ov_vals else None,
        'mean': float(sum(ov_vals) / len(ov_vals)) if ov_vals else None,
        'within_range_0_1': (all(0.0 <= v <= 1.0 for v in ov_vals) if ov_vals else True),
    },
    'split_disjointness': {
        'val_split_ratio': float(VAL_SPLIT_RATIO),
        'train_scene_count': len(train_scene_ids),
        'val_scene_count': len(val_scene_ids),
        'intersection_count': len(intersection),
        'intersection_examples': intersection[:10],
    },
}

report_path = Path(REPO_DIR) / 'runs' / 'megadepth_raw' / 'dataset_report.json'
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(report, indent=2, sort_keys=True))

print('scene dirs:', len(scene_dirs))
print('img dirs  :', len(img_dirs))
print('depth dirs:', len(depth_dirs))
print('image files:', len(img_paths))
print('depth files:', len(depth_paths))
print('corrupt sampled images:', len(corrupt_images))
print('bad sampled depth files:', len(bad_depth_files))
print('overlap sanity:', report['overlap_sanity'])
print('split intersection count:', len(intersection))
print('Dataset report saved to', report_path)

if len(img_dirs) == 0 or len(img_paths) == 0:
    raise RuntimeError('No training images found under **/dense0/imgs. Check RAW_ROOT.')
if corrupt_images:
    raise RuntimeError('Corrupt images detected in sampled validation. See dataset_report.json for details.')
if bad_depth_files:
    print('⚠️ Some sampled depth files failed readability checks; homography fallback may be used for affected pairs.')



In [ ]:
#@title 8) Training preflight import + strict dummy forward checks (CPU/GPU)
import os
import sys
import importlib
import torch

env_paths = [REPO_DIR, XFEAT_DIR, SUPERPOINT_DIR, os.environ.get('PYTHONPATH', '')]
os.environ['PYTHONPATH'] = ':'.join([p for p in env_paths if p]).strip(':')
for p in [REPO_DIR, XFEAT_DIR, SUPERPOINT_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

for module_name, attr in [
    ('modules.xfeat', 'XFeat'),
    ('models.xfeat_core', 'XFeatCore'),
]:
    try:
        mod = importlib.import_module(module_name)
        if hasattr(mod, attr):
            print('XFeat import ok via', module_name, attr)
            break
    except Exception as e:
        last_xfeat_error = e
else:
    raise RuntimeError(f'No compatible XFeat import path found. Last error: {last_xfeat_error}')

for module_name, attr in [
    ('models.superpoint_core', 'SuperPointCore'),
    ('superpoint.superpoint', 'SuperPoint'),
    ('superpoint_pytorch', 'SuperPoint'),
    ('superpoint', 'SuperPoint'),
]:
    try:
        mod = importlib.import_module(module_name)
        if hasattr(mod, attr):
            print('SuperPoint import ok via', module_name, attr)
            break
    except Exception as e:
        last_sp_error = e
else:
    raise RuntimeError(f'No compatible SuperPoint import path found. Last error: {last_sp_error}')

import train

cfg = train.DEFAULT_CONFIG.copy()
cfg.update({
    'mode': 'megadepth_raw',
    'data_root': RAW_ROOT,
    'batch_size': 1,
    'num_workers': 0,
    'image_height': 64,
    'image_width': 64,
    'mixed_precision': False,
    'seed': int(SEED),
})

required_keys = ['keypoints', 'descriptors', 'keypoints_px', 'scores']

for dev_name in (['cpu'] + (['cuda'] if torch.cuda.is_available() else [])):
    device = torch.device(dev_name)
    model = train.build_model(cfg, device)
    x = torch.rand(1, 1, cfg['image_height'], cfg['image_width'], device=device)
    with torch.no_grad():
        out = model.forward_train(x)

    missing = [k for k in required_keys if k not in out]
    if missing:
        raise RuntimeError(f'forward_train missing keys on {device}: {missing}')

    train._validate_forward_output_keys_shapes(out, batch_size=1)
    adapter = out.get('xfeat_adapter_path', 'unknown')
    print(f'Forward preflight passed on {device}; adapter={adapter}; desc_dtype={out["descriptors"][0].dtype}')

print('Preflight checks passed. Ready to train.')



In [ ]:
#@title 9) Select resume checkpoint (auto-pick if empty)
from pathlib import Path

resume = RESUME_CKPT.strip()
if resume and not Path(resume).exists():
    raise FileNotFoundError(f'RESUME_CKPT does not exist: {resume}')

if not resume:
    ckpt_dir = Path(DRIVE_CKPT_DIR)
    best = ckpt_dir / 'best.pth'
    if best.exists():
        resume = str(best)
    else:
        epochs = sorted(ckpt_dir.glob('epoch_*.pth'))
        if epochs:
            resume = str(epochs[-1])

print('Using resume checkpoint:', resume if resume else '(none)')


In [ ]:
#@title 10) Write explicit training override config
import yaml
from pathlib import Path

base_cfg_path = Path(REPO_DIR) / 'config.yaml'
base_cfg = {}
if base_cfg_path.exists():
    with open(base_cfg_path, 'r') as f:
        base_cfg = yaml.safe_load(f) or {}

override_cfg = dict(base_cfg)
override_cfg.update({
    'mode': 'megadepth_raw',
    'data_root': RAW_ROOT,
    'checkpoint_dir': DRIVE_CKPT_DIR,
    'log_dir': DRIVE_LOG_DIR,
    'batch_size': int(BATCH_SIZE),
    'max_epochs': int(MAX_EPOCHS),
    'lr': float(LR),
    'num_workers': int(NUM_WORKERS),
    'dataloader_timeout_s': 0,
    'seed': int(SEED),
    'image_height': int(IMAGE_H),
    'image_width': int(IMAGE_W),
    'megadepth_val_split_ratio': float(VAL_SPLIT_RATIO),
    'max_pairs_per_scene': int(MAX_PAIRS_PER_SCENE),
    'min_overlap': float(MIN_OVERLAP),
    'max_overlap': float(MAX_OVERLAP),
    'min_keypoint_score': float(MIN_KEYPOINT_SCORE),
    'verify_dataset_pairs': (not bool(NO_VERIFY_DATASET_PAIRS)),
})

override_cfg_path = Path(OVERRIDE_CFG_PATH)
override_cfg_path.write_text(yaml.safe_dump(override_cfg, sort_keys=False))
print('Wrote config override to', override_cfg_path)
print(override_cfg_path.read_text())



In [ ]:
#@title 11) Start TensorBoard (live metrics)
%load_ext tensorboard
%tensorboard --logdir $DRIVE_LOG_DIR --host 127.0.0.1 --port 6006


In [ ]:
#@title 12) Launch MegaDepth Raw training (stream logs)
import os
import sys
import subprocess

env = os.environ.copy()
env['PYTHONPATH'] = ':'.join([REPO_DIR, XFEAT_DIR, SUPERPOINT_DIR, env.get('PYTHONPATH', '')]).strip(':')
env['PYTHONUNBUFFERED'] = '1'

cmd = [
    sys.executable, f'{REPO_DIR}/train.py',
    '--config', str(OVERRIDE_CFG_PATH),
]
if NO_VERIFY_DATASET_PAIRS:
    cmd.append('--no_verify_dataset_pairs')
if resume:
    cmd += ['--resume', resume]

print('Running command\n', ' '.join(cmd))
proc = subprocess.Popen(
    cmd,
    cwd=REPO_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end='')

ret = proc.wait()
if ret != 0:
    raise RuntimeError(f'Training failed with exit code {ret}')

print('Training finished successfully.')


In [ ]:
#@title 13) List saved checkpoints and logs
from pathlib import Path

ckpt_dir = Path(DRIVE_CKPT_DIR)
log_dir = Path(DRIVE_LOG_DIR)
ckpts = sorted(ckpt_dir.glob('*.pth'))

print('Checkpoint dir:', ckpt_dir)
print('Found', len(ckpts), 'checkpoint files')
for p in ckpts[-30:]:
    print('-', p.name)

print('\nTensorBoard log dir:', log_dir)
if log_dir.exists():
    event_files = sorted(log_dir.glob('**/events.out.tfevents.*'))
    print('Event files:', len(event_files))
else:
    print('Log directory does not exist yet.')
